## pipeline

`pipeline`関数はTransformersの機能を最も簡易的に実行できるAPIです

例えば`pipeline("タスク名")`のように最初に引数でタスク名を指定し、推論タスクを簡単に実行できるインスタンスが作成できます。

```python
classifier = pipeline("sentiment-analysis")
```

また、`pipeline(タスク名, model=モデル名)`のように`model`引数でモデルを指定できます（モデル名を指定しないと、"distilgpt2"という小さなGPTモデルが使用される）。

```python
classifier = pipeline("sentiment-analysis", model="distilgpt2")
```

### タスクの指定

`pipeline("タスク名")`のタスク名に以下のような文字列を指定することで、推論タスクを簡単に実行できるインスタンスが作成できます

- "feature-extraction"：テキストのベクトル表現を取得
- "fill-mask"：マスクされたトークンの予測（穴埋め）
- "ner"：固有表現認識
- "question-answering"：質問応答（文脈に基づく回答抽出）
- "sentiment-analysis"：感情分析
- "summarization"：要約生成
- "text-generation"：テキスト生成（文章生成）
- "translation"：機械翻訳
- "zero-shot-classification"：ゼロショット分類（未学習ラベルでの分類）

#### 感情分析

例えば"sentiment-analysis"（感情分析）は以下のように実行できます。

In [ ]:
# パイプライン実行例（感情分析）
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace course my whole life.")

pipelineの推論は複数データをリスト指定して一括実行もできます。

In [ ]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

#### ゼロショット分類

ラベル付けされていないテキストを分類

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

### テキスト生成

テキスト生成（後に続く文章を予測する）を使用したい場合、`pipeline("text-generation")`のように指定できます。

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

以下のような引数を指定することで、返答のパターンを変えることができます

- "max_length"引数：入力＋出力を合わせたトークン数の上限（実用上は出力トークンの上限を指定する"max_new_tokens"の方が使いやすい）
- "num_return_sequences"引数：複数パターンのテキスト生成

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

#### マスクされたトークンの予測（穴埋め）

`top_k`引数で推論出力する予測候補の数を指定します

In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

#### 固有表現認識

In [ ]:
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

#### 質問応答（transformers v5系の代替例）

`question-answering`タスク名はv5系で利用できないため、ここでは`text-generation`で文脈つき質問を実行します。

In [ ]:
from transformers import pipeline

qa_generator = pipeline("text-generation")
prompt = (
    "Context: My name is Sylvain and I work at Hugging Face in Brooklyn.\n"
    "Question: Where do I work?\n"
    "Answer:"
)
qa_generator(prompt, max_new_tokens=20, do_sample=False)

#### 要約（transformers v5系の代替例）

`summarization`タスク名はv5系で利用できないため、ここでは`text-generation`に要約指示を与えて実行します。

In [ ]:
from transformers import pipeline

summarizer = pipeline("text-generation", model="distilgpt2")
text = """
America has changed dramatically during recent years. Not only has the number of 
graduates in traditional engineering disciplines such as mechanical, civil, 
electrical, chemical, and aeronautical engineering declined, but in most of 
the premier American universities engineering curricula now concentrate on 
and encourage largely the study of engineering science. As a result, there 
are declining offerings in engineering subjects dealing with infrastructure, 
the environment, and related issues, and greater concentration on high 
technology subjects, largely supporting increasingly complex scientific 
developments. While the latter is important, it should not be at the expense 
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other 
industrial countries in Europe and Asia, continue to encourage and advance 
the teaching of engineering. Both China and India, respectively, graduate 
six and eight times as many traditional engineers as does the United States. 
Other industrial countries at minimum maintain their output, while America 
suffers an increasingly serious decline in the number of engineering graduates 
and a lack of well-educated engineers.
"""

prompt = f"Summarize the following text in 2 concise sentences:\n\n{text}\n\nSummary:"
summarizer(prompt, max_new_tokens=80, do_sample=False, return_full_text=False)

### Hugging Faceのタグ名で見るべきポイント

モデルを探すときは、[公式](https://huggingface.co/models)でタグ名検索するのが基本。

例えば`Llama-3.2-3B-Instruct`

- `Llama`: モデル名
- `3.2`: モデルのバージョン
- `3B`: モデルのサイズ
- `Instruct`: 対応タスク

モデルのサイズと対応タスクについてもう少し詳細に解説します

#### モデルのサイズ

モデルのサイズと必要VRAMは、大まかに以下のような関係があります

- FP16：約 2GB / 1Bパラメータ
- INT8：約 1GB / 1Bパラメータ
- 4bit（量子化）：約 0.5GB / 1Bパラメータ

実際はKVキャッシュやオーバーヘッドで上記よりもVRAMを使うので、30%ほど余裕を見て以下くらいが目安になると思います

| モデルサイズ | FP16での必要VRAM    | 4bit量子化での必要VRAM |
| ------ | ------------------- | -------- |
| 3B     | 6GB → **8GB程度必要**   | 2GB      |
| 7B     | 14GB → **16GB必要**   | 4GB      |
| 13B    | 26GB → **30GB必要**   | 8GB      |
| 70B    | 140GB → **複数GPU必須** | 35GB     |

#### 対応タスク

||タグ名の例|`Tasks`の選択肢|例|用途|
|---|---|---|---|---|
|Baseモデル|(なし)|`Text Generation`等|"google/gemma-2-2b"|研究・事前学習（自分でファインチューニングが必要）|
|Instructモデル|`Instruct`、`it`、`Chat`等|`Text Generation`または`Image-Text-to-Text`|"google/gemma-2-2b-it"|対話（Instruction Tuningされている）|
|Visionモデル|`Vision`、`VL`等（最近はタグのないVision対応モデルが多い）|`Image-Text-to-Text`|"google/gemma-2-2b-it"|画像解釈|

チャットのようなテキスト生成をしたいのであれば、とりあえず`Instruct`にしておけばOK（Gemmaだと`it`）なので、検索時に`Tasks`の`Text Generation`（LLM）または`Image-Text-to-Text`（VLM）を選択すればOKです（最近はデフォルトでVLMになっているモデルが多いので、テキスト生成でも`Image-Text-to-Text`が適切なケースもあります）。

### おすすめモデル

Hugging Face内の使いやすいInstructionモデル（Instruction Tuningされたモデル）のうち、いいねが多いものを、VRAMサイズごとに以下に示します（同モデルでも量子化の有無で重複があることにご注意ください）。

#### 8GB VRAM (RTX5060等)

量子化なしで3Bくらい、量子化ありで10Bくらいが上限

|`model`引数に指定する文字列|いいね数|備考|
|---|---|---|
|"microsoft/Phi-3-mini-4k-instruct"|1.41k||
|"Qwen/Qwen2.5-1.5B-Instruct"|659||
|"google/gemma-2-2b-it"|1.33k||
|"meta-llama/Llama-3.2-3B-Instruct"|2.09k||
|"mistralai/Mistral-7B-Instruct-v0.2"|3.11k|4bit量子化必要|
|"Qwen/Qwen2.5-7B-Instruct"|1.19k|4bit量子化必要|
|"meta-llama/Llama-3.1-8B-Instruct"|5.67k|4bit量子化必要|

#### 24GB VRAM (RTX4090等)

量子化なしで10Bくらい、量子化ありで30Bくらいが上限

|`model`引数に指定する文字列|いいね数|備考|
|---|---|---|
|"mistralai/Mistral-7B-Instruct-v0.2"|3.11k||
|"Qwen/Qwen2.5-7B-Instruct"|1.19k||
|"meta-llama/Llama-3.1-8B-Instruct"|5.67k||
|"google/gemma-3-12b-it"|703||
|"google/gemma-3-27b-it"|1.95k|4bit量子化必要|
|"google/gemma-4-31b-it"|1.53k|4bit量子化必要|

#### 96GB VRAM (RTX PRO 6000 Blackwell等)

量子化なしで40Bくらい、量子化ありで120Bくらいが上限

|`model`引数に指定する文字列|いいね数|備考|
|---|---|---|
|"google/gemma-3-27b-it"|1.95k||
|"google/gemma-4-31b-it"|1.53k||
|"meta-llama/Llama-3.3-70B-Instruct"|2.69k|4bit量子化or複数GPU必要|
|"Qwen/Qwen2.5-72B-Instruct"|925|4bit量子化or複数GPU必要|